# transistor：正常画像の復元に適したVAEモデルの比較
2026-11-06作成。**コード作成のみ・未実行です。実行、学習、評価はユーザーが行います。**

回転・反復拡張を使わず、FC、CNN Shallow3、CNN Deep5、Hybrid CNN＋FCの4種類を同じ条件で学習します。既存モデルの重みは流用しません。Sparse VAEではありません。

| 項目 | 共通条件 |
| --- | --- |
| 入力と復元目標 | 元画像全体1024×1024 → LANCZOSで128×128 → 白黒・0〜1 |
| データ | train/goodの正常213枚を、学習170枚・検証43枚に固定分割 |
| 分割seed／学習seed | 42／42 |
| 学習 | 100epoch、batch32、Adam、lr0.001、潜在64 |
| β | min(1, zero_based_epoch/50)。表示epoch1で0、epoch51で1 |
| 更新回数 | ceil(170/32)×100＝600回（全モデル共通） |
| 選定 | 最終100epochの検証MSEを主基準。SSIMと見た目を補助にする |

**最初の学習：** 初期設定は `RUN_FULL_TRAINING = True` です。上からすべて実行すると4モデルを学習し、比較図を表示します。

**保存済み結果だけを見る：** 最初のセルを `RUN_FULL_TRAINING = False` にし、`TRAINING_RUN_ID`に実行IDを指定して上から実行します。Noneは最新の完了済み実行です。Trueのまま全セルを再実行すると、新しい実行として再び学習します。

テスト画像は読み込みません。検証43枚はモデル選定用であり、最終テストではありません。通常画像をきれいに復元できても、異常検知のAUROCが高いとは限りません。この段階では異常検知スコア・閾値・AUROCは計算しません。

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from io import BytesIO
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import json
import math
import platform
import random
import time
import uuid
import importlib.metadata
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from skimage.metrics import structural_similarity
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'memory/memory.md').is_file() and (p / 'data/transistor/train/good').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('リポジトリ内を作業ディレクトリにしてください。')
DATA_DIR = ROOT / 'data/transistor/train/good'
OUTPUT_BASE = ROOT / 'saved_models/transistor_reconstruction_models_20261106'
RUN_FULL_TRAINING = True
TRAINING_RUN_ID = None  # 保存済みモデルを見る場合はRUN_FULL_TRAINING=Falseにする。
SPLIT_SEED = 42
TRAIN_SEED = 42
VALIDATION_COUNT = 43
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 0.001
LATENT_DIM = 64
ANNEALING_EPOCHS = 50
MODEL_KEYS = ('FC', 'Shallow3', 'Deep5', 'Hybrid')
LABELS = {'FC': 'FC', 'Shallow3': 'CNN Shallow3', 'Deep5': 'CNN Deep5', 'Hybrid': 'Hybrid CNN＋FC'}
COLORS = {'FC': '#777744', 'Shallow3': '#287c80', 'Deep5': '#446c9b', 'Hybrid': '#b45342'}
EXPERIMENT = 'transistor_normal_reconstruction_four_vaes_20261106_v1'
ARCHITECTURE_VERSION = 1
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0
SSIM_WINDOW = 11
MAX_RECON_IMAGES = 6  # Noneで検証43枚すべて。表示・保存する図が多くなります。
PREVIEW_IMAGES = 2
LEAD_ROI = (24, 64, 112, 128)  # 128入力上の(x0,y0,x1,y1)。端子に合わせて変更可能。
ABS_ERROR_VMAX = 1.0  # 全モデル共通。差分は画素値の絶対差で、SSIMマップではない。
assert EPOCHS > 0 and BATCH_SIZE >= 2 and ANNEALING_EPOCHS > 0
assert 0 < ABS_ERROR_VMAX <= 1
if MAX_RECON_IMAGES is not None:
    assert isinstance(MAX_RECON_IMAGES, int) and not isinstance(MAX_RECON_IMAGES, bool) and MAX_RECON_IMAGES > 0
rx0, ry0, rx1, ry1 = LEAD_ROI
assert 0 <= rx0 < rx1 <= 128 and 0 <= ry0 < ry1 <= 128

torch.set_num_threads(min(4, max(1, torch.get_num_threads())))
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
fonts = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams.update({'font.family': next((f for f in ['Meiryo', 'Yu Gothic', 'Noto Sans CJK JP']
                                         if f in fonts), 'DejaVu Sans'),
                     'font.size': 11, 'figure.dpi': 100, 'axes.unicode_minus': False})

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f'デバイス：{DEVICE} / 正式学習：{RUN_FULL_TRAINING} / 乱数seed：{TRAIN_SEED}')

## 1. 正常画像だけを固定分割する
全モデルで同じ170枚から学習し、同じ43枚で検証します。検証画像は勾配計算・学習には使いません。過去の213枚すべてを学習した実験とは分割が異なるため、過去の結果との直接比較にはしません。

分割はファイル名順の一覧に固定乱数の並べ替えを適用して決め、各分割内はファイル名順に戻します。ファイル・前処理後の画素値のSHA-256も保存します。

In [ ]:
paths = sorted(DATA_DIR.glob('*.png'), key=lambda p: p.name)
if len(paths) != 213:
    raise ValueError(f'想定した正常画像213枚と異なります：{len(paths)}枚')
inputs, manifest = [], []
for path in paths:
    raw = path.read_bytes()
    with Image.open(BytesIO(raw)) as im:
        rgb = im.convert('RGB')
        if rgb.size != (1024, 1024):
            raise ValueError(f'想定外の画像サイズ：{path} {rgb.size}')
        small = rgb.resize((128, 128), resample=Image.Resampling.LANCZOS).convert('L')
        array = np.asarray(small, dtype=np.float32) / np.float32(255)
    inputs.append(array)
    manifest.append({'path': path.relative_to(ROOT).as_posix(), 'source_size': [1024, 1024],
                     'file_sha256': hashlib.sha256(raw).hexdigest(),
                     'preprocessed_sha256': hashlib.sha256(array.tobytes()).hexdigest()})
all_inputs = torch.from_numpy(np.stack(inputs)[:, None]).contiguous()
assert tuple(all_inputs.shape) == (213, 1, 128, 128)
assert torch.isfinite(all_inputs).all() and 0 <= all_inputs.min() <= all_inputs.max() <= 1
permutation = np.random.default_rng(SPLIT_SEED).permutation(len(paths))
validation_indices = np.sort(permutation[:VALIDATION_COUNT])
train_indices = np.sort(permutation[VALIDATION_COUNT:])
assert len(train_indices) == 170 and len(validation_indices) == 43
assert not set(train_indices) & set(validation_indices)
train_tensor = all_inputs[train_indices].contiguous()
validation_tensor = all_inputs[validation_indices].contiguous()
train_manifest = [manifest[int(i)] for i in train_indices]
validation_manifest = [manifest[int(i)] for i in validation_indices]
assert not {r['file_sha256'] for r in train_manifest} & {r['file_sha256'] for r in validation_manifest}
assert not {r['preprocessed_sha256'] for r in train_manifest} & {r['preprocessed_sha256'] for r in validation_manifest}, '同じ画素の画像が分割をまたいでいます。'
split_record = {'split_seed': SPLIT_SEED, 'train': train_manifest, 'validation': validation_manifest}
data_signature = hashlib.sha256(json.dumps(split_record, ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()
UPDATES_PER_EPOCH = math.ceil(len(train_tensor) / BATCH_SIZE)
EXPECTED_UPDATES = EPOCHS * UPDATES_PER_EPOCH
if len(train_tensor) % BATCH_SIZE == 1:
    raise ValueError('BatchNormのため最終バッチ1枚になる設定を避けてください。')
display(pd.DataFrame([{'分割': '学習', '枚数': len(train_tensor)},
                      {'分割': 'モデル選定用の検証', '枚数': len(validation_tensor)}]))
print(f'全モデル共通：{UPDATES_PER_EPOCH}更新/epoch、{EXPECTED_UPDATES}更新/モデル')
print('分割・データ識別情報：', data_signature)

## 2. 復元目標と、縮小時点で残る細部を確認する
モデルの復元目標は128×128画像です。元の1024画像と同じ細部が残るとは限りません。下の図で、元画像、入力画像、同じ端子領域の拡大を見比べます。緑枠は固定の表示範囲であり、端子の正解マスクではありません。

In [ ]:
for index in range(min(PREVIEW_IMAGES, len(validation_manifest))):
    record = validation_manifest[index]
    with Image.open(ROOT / record['path']) as im:
        raw_gray = np.asarray(im.convert('L'), dtype=np.float32) / np.float32(255)
    small = validation_tensor[index, 0].numpy()
    panels = [raw_gray, small, raw_gray[ry0 * 8:ry1 * 8, rx0 * 8:rx1 * 8], small[ry0:ry1, rx0:rx1]]
    titles = ['元画像（白黒1024×1024）', '復元目標（128×128）', '元画像の端子領域', '128入力の同じ領域']
    fig, axes = plt.subplots(1, 4, figsize=(15, 4), constrained_layout=True)
    for col, (ax, array, title) in enumerate(zip(axes, panels, titles)):
        ax.imshow(array, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        ax.set_title(title)
        ax.axis('off')
        if col < 2:
            scale = 8 if col == 0 else 1
            ax.add_patch(Rectangle((rx0 * scale - .5, ry0 * scale - .5),
                                    (rx1 - rx0) * scale, (ry1 - ry0) * scale,
                                    fill=False, edgecolor='#22aa66', linewidth=2))
    fig.suptitle(record['path'], fontsize=12)
    plt.show()
    plt.close(fig)

## 3. 比較する4種類のVAE
層構成は既存のFC・Shallow3・Deep5・Hybridを基にしています。すべて学習時はサンプリング、復元の検証時は平均μを使用します。数値安定化のlog分散制限は全モデルで−10〜10に統一しています（過去のFCにはなかった制限を追加）。

構造が異なるため、初期重みの値は同一にはしません。初期化seedとデータの順序をそろえます。重みやパラメータ数が大きいほど、必ず復元がよくなるわけではありません。

In [ ]:
class VAE_FC(nn.Module):
    """
    全結合層(FC)のみを用いて特徴抽出と復元を行うベースラインモデル
    """
    def __init__(self, latent_dim=64):
        super(VAE_FC, self).__init__()
        
        # --- エンコーダ: 段階的に次元を圧縮 (16384 -> 1024 -> 256 -> 128) ---
        self.encoder = nn.Sequential(
            nn.Linear(16384, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )
        
        # 潜在空間へのマッピング
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # --- デコーダ: 段階的に次元を復元 (64 -> 128 -> 256 -> 1024 -> 16384) ---
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            nn.Linear(1024, 16384), nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        # 画像を1次元に平坦化
        h = torch.flatten(x, start_dim=1)
        h = self.encoder(h)
        
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu
        
        recon = self.decoder(z)
        
        # 画像の形状に戻して返す
        return recon.view(-1, 1, 128, 128), mu, logvar


class VAE_CNN_Shallow3(nn.Module):
    """
    浅い3層の畳み込み層を用いた軽量VAEモデル
    """
    def __init__(self, latent_dim=64):
        super(VAE_CNN_Shallow3, self).__init__()
        
        # --- エンコーダ: 畳み込み層で空間情報を保ちながら圧縮 (128 -> 64 -> 32 -> 16) ---
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        
        # 64チャンネル * 16 * 16 = 16384次元
        self.fc_mu = nn.Linear(64 * 16 * 16, latent_dim)
        self.fc_logvar = nn.Linear(64 * 16 * 16, latent_dim)
        
        self.decoder_fc = nn.Linear(latent_dim, 64 * 16 * 16)
        
        # --- デコーダ: 逆畳み込み層で元の画像サイズへ復元 ---
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)
        
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        
        # 潜在変数の発散(KLD爆発)を防ぐ安全装置
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu
        
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 64, 16, 16) 
        recon = self.decoder_conv(h_recon)
        
        return recon, mu, logvar


class VAE_CNN_Deep5(nn.Module):
    """
    5層の深い畳み込み層を用いたVAEモデル
    """
    def __init__(self, latent_dim=64):
        super(VAE_CNN_Deep5, self).__init__()
        
        # --- エンコーダ: 畳み込み層を5層重ねてさらに深く圧縮 (128 -> 64 -> 32 -> 16 -> 8 -> 4) ---
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        )
        
        # 最終層のサイズ: 256チャンネル * 4ピクセル * 4ピクセル = 4096次元
        self.fc_mu = nn.Linear(256 * 4 * 4, latent_dim)
        self.fc_logvar = nn.Linear(256 * 4 * 4, latent_dim)
        
        # デコーダ用の全結合層
        self.decoder_fc = nn.Linear(latent_dim, 256 * 4 * 4)
        
        # --- デコーダ: 逆畳み込み層を5層重ねて画像サイズを復元 (4 -> 8 -> 16 -> 32 -> 64 -> 128) ---
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)
        
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        
        # 潜在変数の発散(KLD爆発)を防ぐ安全装置
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)
        
        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu
        
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 256, 4, 4) 
        recon = self.decoder_conv(h_recon)
        
        return recon, mu, logvar


class VAE_Hybrid_CNN_FC(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_Hybrid_CNN_FC, self).__init__()

        # -------------------------------------------------------------
        # 【エンコーダ前半】 CNN_Deep5（空間圧縮: 128x128 -> 4x4）
        # -------------------------------------------------------------
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【エンコーダ後半】 FC_Only（段階的圧縮: 4096次元 -> 128次元）
        # -------------------------------------------------------------
        self.encoder_fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )

        # 潜在空間へのマッピング（128次元 -> 64次元）
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # -------------------------------------------------------------
        # 【デコーダ前半】 FC_Only（段階的復元: 64次元 -> 4096次元）
        # -------------------------------------------------------------
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            # ※CNNへ繋ぐため、最後はSigmoidではなくReLUを使用します
            nn.Linear(1024, 256 * 4 * 4), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【デコーダ後半】 CNN_Deep5（空間復元: 4x4 -> 128x128）
        # -------------------------------------------------------------
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        # 1. CNNによる特徴抽出
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)

        # 2. FCによる段階的圧縮
        h = self.encoder_fc(h)

        # 3. 潜在変数の計算と安全装置（クランプ）
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)

        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu

        # 4. FCによる段階的復元
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 256, 4, 4) # 4階テンソルに戻す

        # 5. CNNによる画像復元
        recon = self.decoder_conv(h_recon)

        return recon, mu, logvar

MODEL_FACTORIES = {'FC': VAE_FC, 'Shallow3': VAE_CNN_Shallow3,
                   'Deep5': VAE_CNN_Deep5, 'Hybrid': VAE_Hybrid_CNN_FC}
MODEL_DESCRIPTIONS = {
    'FC': '全結合：16384→1024→256→128→潜在64',
    'Shallow3': 'CNN3段：128→64→32→16、最終特徴を潜在64へ',
    'Deep5': 'CNN5段：128→64→32→16→8→4、最終特徴を潜在64へ',
    'Hybrid': 'CNN5段＋全結合：4096→1024→256→128→潜在64',
}

In [ ]:
model_rows = []
for key in MODEL_KEYS:
    seed_everything(TRAIN_SEED)
    model = MODEL_FACTORIES[key](latent_dim=LATENT_DIM)
    model_rows.append({'model': key, '表示名': LABELS[key], '構造': MODEL_DESCRIPTIONS[key],
                       'パラメータ数': sum(p.numel() for p in model.parameters() if p.requires_grad),
                       '潜在次元': LATENT_DIM, '共通更新回数': EXPECTED_UPDATES})
    del model
model_table = pd.DataFrame(model_rows)
display(model_table)

## 4. 学習・検証の共通処理
勾配に使う損失は、画素の二乗誤差合計SSE＋β×標準正規分布へのKL合計です。ログは各値を画像数で割ります。MSEはSSEを128×128で割った1画素あたりの値です。

学習はサンプリングを使い、検証は平均μとevalモードを使います。そのため学習曲線と検証曲線の差には推論方法やBatchNormのモードの違いも含まれます。単純に差のすべてを過学習と判断しません。

毎epochで検証MSEを記録しますが、早期終了や最良epochの選択はせず、全モデルの最終100epoch（600更新）を比較します。SSIMは最終評価時に計算します。

In [ ]:
def copy_cpu_state(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def state_digest(state):
    digest = hashlib.sha256()
    for name in sorted(state):
        value = state[name].detach().cpu().contiguous()
        digest.update(name.encode('utf-8'))
        digest.update(str(value.dtype).encode('ascii'))
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def vae_loss(reconstruction, inputs, mu, logvar, beta):
    sse = nn.functional.mse_loss(reconstruction, inputs, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.square() - logvar.exp())
    return sse, kl, sse + beta * kl


def make_loader():
    generator = torch.Generator().manual_seed(TRAIN_SEED + 1000)
    dataset = TensorDataset(train_tensor, torch.arange(len(train_tensor), dtype=torch.int64))
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
                      generator=generator, num_workers=NUM_WORKERS)


def train_epoch(model, loader, optimizer, beta, order_digest):
    model.train()
    sums = np.zeros(3, dtype=np.float64)
    seen = updates = 0
    for inputs, indices in loader:
        order_digest.update(indices.numpy().astype('<i8', copy=False).tobytes())
        inputs = inputs.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        reconstruction, mu, logvar = model(inputs, sample=True)
        sse, kl, total = vae_loss(reconstruction, inputs, mu, logvar, beta)
        if not all(torch.isfinite(v).item() for v in (sse, kl, total)):
            raise FloatingPointError('学習損失にNaN/infが発生しました。')
        total.backward()
        if any(p.grad is not None and not torch.isfinite(p.grad).all().item() for p in model.parameters()):
            raise FloatingPointError('勾配にNaN/infが発生しました。')
        optimizer.step()
        sums += [sse.item(), kl.item(), total.item()]
        seen += len(inputs)
        updates += 1
    if seen != len(train_tensor) or updates != UPDATES_PER_EPOCH:
        raise AssertionError('共通の学習枚数・更新回数と一致しません。')
    return {'train_sse_per_image': float(sums[0] / seen),
            'train_mse': float(sums[0] / seen / (128 * 128)),
            'train_kl_per_image': float(sums[1] / seen),
            'train_total_per_image': float(sums[2] / seen),
            'seen_images': seen, 'updates': updates}


def reconstruct_normal(model, tensor, include_ssim=False):
    model.eval()
    count = len(tensor)
    restored = np.empty((count, 128, 128), dtype=np.float32)
    sse_values = np.empty(count, dtype=np.float64)
    kl_values = np.empty(count, dtype=np.float64)
    with torch.inference_mode():
        for start in range(0, count, BATCH_SIZE):
            inputs = tensor[start:start + BATCH_SIZE].to(DEVICE)
            reconstruction, mu, logvar = model(inputs, sample=False)
            if reconstruction.shape != inputs.shape or not torch.isfinite(reconstruction).all().item():
                raise FloatingPointError('復元画像の形状・値が不正です。')
            sse = (inputs - reconstruction).square().flatten(1).sum(1)
            kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(1)
            stop = start + len(inputs)
            restored[start:stop] = reconstruction[:, 0].cpu().numpy()
            sse_values[start:stop] = sse.cpu().numpy()
            kl_values[start:stop] = kl.cpu().numpy()
    if not np.isfinite(sse_values).all() or not np.isfinite(kl_values).all():
        raise FloatingPointError('検証誤差・KLにNaN/infが発生しました。')
    values = {'reconstruction': restored, 'sse': sse_values,
              'mse': sse_values / (128 * 128), 'kl': kl_values}
    if include_ssim:
        originals = tensor[:, 0].cpu().numpy()
        values['ssim'] = np.asarray([structural_similarity(x, y, data_range=1.0, win_size=SSIM_WINDOW)
                                     for x, y in zip(originals, restored)], dtype=np.float64)
        if not np.isfinite(values['ssim']).all():
            raise FloatingPointError('SSIMにNaN/infが発生しました。')
    return values


def validation_frame(values):
    return pd.DataFrame({'path': [r['path'] for r in validation_manifest],
                         'sse': values['sse'], 'mse': values['mse'],
                         'ssim': values['ssim'], 'kl': values['kl']})


def train_model(key, directory):
    directory.mkdir(exist_ok=False)
    seed_everything(TRAIN_SEED)
    model = MODEL_FACTORIES[key](latent_dim=LATENT_DIM).to(DEVICE)
    initial_hash = state_digest(model.state_dict())
    # 構造ごとの重み初期化で消費する乱数の差と、学習中の乱数を分ける。
    seed_everything(TRAIN_SEED + 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loader = make_loader()  # 各モデルで同じseedのgeneratorを新規作成する。
    order_digest = hashlib.sha256()
    history = []
    started = time.perf_counter()
    (directory / 'architecture.txt').write_text(str(model), encoding='utf-8')
    for epoch in range(EPOCHS):
        beta = min(1.0, epoch / ANNEALING_EPOCHS)
        values = train_epoch(model, loader, optimizer, beta, order_digest)
        validation = reconstruct_normal(model, validation_tensor)
        history.append({'epoch': epoch + 1, 'beta': beta,
                        'cumulative_updates': (epoch + 1) * UPDATES_PER_EPOCH,
                        'val_sse_per_image': float(validation['sse'].mean()),
                        'val_mse': float(validation['mse'].mean()),
                        'val_kl_per_image': float(validation['kl'].mean()), **values})
        pd.DataFrame(history).to_csv(directory / 'history.csv', index=False)
        if epoch == 0 or (epoch + 1) % 10 == 0 or epoch + 1 == EPOCHS:
            print(f'{LABELS[key]} | epoch {epoch + 1}/{EPOCHS} | beta={beta:.2f} | '
                  f'train MSE={values["train_mse"]:.6f} | val MSE={history[-1]["val_mse"]:.6f}', flush=True)
    final = reconstruct_normal(model, validation_tensor, include_ssim=True)
    weight_path = directory / 'model_final.pth'
    torch.save(copy_cpu_state(model), weight_path)
    validation_frame(final).to_csv(directory / 'validation_final.csv', index=False)
    summary = {'model': key, 'epochs': EPOCHS, 'updates': EPOCHS * UPDATES_PER_EPOCH,
               'split_seed': SPLIT_SEED, 'train_seed': TRAIN_SEED,
               'initial_state_sha256': initial_hash, 'sample_order_sha256': order_digest.hexdigest(),
               'weights_sha256': sha256(weight_path), 'data_signature': data_signature,
               'val_mse': float(final['mse'].mean()), 'val_sse_per_image': float(final['sse'].mean()),
               'val_ssim': float(final['ssim'].mean()), 'val_kl_per_image': float(final['kl'].mean()),
               'seconds': time.perf_counter() - started}
    if summary['updates'] != EXPECTED_UPDATES:
        raise AssertionError('想定した更新回数と異なります。')
    write_json(directory / 'summary.json', summary)
    del model, optimizer, final, validation
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return summary

## 5. 正式学習：4モデルを同じ条件で学習する
初期値はTrueです。実行するたびに新規フォルダを作り、既存のモデルや実験を上書きしません。中断した実行は完了済みのモデル比較として読み込みません。FCなどはパラメータ数が多く、CPUでは時間がかかる場合があります。

In [ ]:
run_dir = None
training_summaries = []
if RUN_FULL_TRAINING:
    run_id = datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
    run_dir = OUTPUT_BASE / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    config = {'experiment': EXPERIMENT, 'architecture_version': ARCHITECTURE_VERSION,
              'run_id': run_id, 'created_jst': datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),
              'dataset': 'transistor/train/good', 'augmentation': 'none', 'model_keys': list(MODEL_KEYS),
              'train_count': len(train_tensor), 'validation_count': len(validation_tensor),
              'split_seed': SPLIT_SEED, 'train_seed': TRAIN_SEED, 'data_signature': data_signature,
              'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'updates_per_epoch': UPDATES_PER_EPOCH,
              'updates_per_model': EXPECTED_UPDATES, 'optimizer': 'Adam', 'learning_rate': LEARNING_RATE,
              'latent_dim': LATENT_DIM, 'annealing_epochs': ANNEALING_EPOCHS,
              'beta_rule': 'min(1.0, zero_based_epoch / annealing_epochs)',
              'preprocessing': 'Full RGB1024 -> Resize128 LANCZOS -> Grayscale -> float32/255; no crop/rotation/padding',
              'input_size': [1, 128, 128], 'logvar_clamp': [-10, 10],
              'training_inference': 'posterior sampling', 'validation_inference': 'posterior mean; eval mode',
              'loss': 'sum pixel SSE + beta * sum standard-normal KL',
              'selection': 'final-epoch validation mean MSE; SSIM and visual inspection supplementary',
              'ssim': {'window': SSIM_WINDOW, 'data_range': 1.0, 'top_k_used': False},
              'test_data_used': False, 'device': str(DEVICE), 'python': platform.python_version(),
              'versions': {p: importlib.metadata.version(p) for p in ['torch', 'numpy', 'pandas', 'Pillow', 'scikit-image', 'matplotlib']},
              'status': 'running'}
    write_json(run_dir / 'config.json', config)
    write_json(run_dir / 'split_manifest.json', split_record)
    model_table.to_csv(run_dir / 'model_table.csv', index=False)
    try:
        for key in MODEL_KEYS:
            summary = train_model(key, run_dir / key)
            training_summaries.append(summary)
            pd.DataFrame(training_summaries).to_csv(run_dir / 'training_summary.csv', index=False)
        if len({s['sample_order_sha256'] for s in training_summaries}) != 1:
            raise AssertionError('モデル間で画像順が一致しません。')
        if any(s['updates'] != EXPECTED_UPDATES for s in training_summaries):
            raise AssertionError('モデル間で更新回数が一致しません。')
        config['status'] = 'completed'
        write_json(run_dir / 'config.json', config)
        print(f'4モデルの学習完了：{run_dir}')
        print(f'保存済み結果を見るときのTRAINING_RUN_ID：{run_id}')
    except BaseException as exc:
        config['status'] = 'interrupted_or_failed'
        config['error'] = f'{type(exc).__name__}: {exc}'
        write_json(run_dir / 'config.json', config)
        raise
else:
    print('学習を省略し、以下で保存済みモデルとログを読み込みます。')

## 6. 保存済みモデルとログの読み込み
再学習せずに比較するときは、先頭のRUN_FULL_TRAININGをFalseにして実行します。TRAINING_RUN_IDがNoneなら、このセルより前に新規学習した実行、または最新の完了済み実行を選びます。

データ分割・画像識別情報・学習設定・モデル種別を照合します。中断モデルや過去の回転実験は読み込みません。Trueのまま「すべて実行」すると新規学習になる点に注意してください。

In [ ]:
completed_runs = []
if OUTPUT_BASE.is_dir():
    for candidate in sorted(OUTPUT_BASE.iterdir(), key=lambda p: p.name):
        if (candidate / 'config.json').is_file():
            saved = read_json(candidate / 'config.json')
            if saved.get('experiment') == EXPERIMENT and saved.get('status') == 'completed':
                completed_runs.append(candidate)
RUN_DIR = (OUTPUT_BASE / TRAINING_RUN_ID if TRAINING_RUN_ID is not None else
           (run_dir if run_dir is not None else (completed_runs[-1] if completed_runs else None)))
READY = RUN_DIR is not None
histories, model_paths, summaries = {}, {}, {}
if READY:
    saved_config = read_json(RUN_DIR / 'config.json')
    if saved_config.get('experiment') != EXPERIMENT or saved_config.get('status') != 'completed':
        raise ValueError('この4モデル比較の完了済み実行を指定してください。')
    expected = {'architecture_version': ARCHITECTURE_VERSION, 'model_keys': list(MODEL_KEYS),
                'augmentation': 'none', 'train_count': 170, 'validation_count': 43,
                'data_signature': data_signature, 'split_seed': SPLIT_SEED, 'train_seed': TRAIN_SEED,
                'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'latent_dim': LATENT_DIM,
                'learning_rate': LEARNING_RATE, 'annealing_epochs': ANNEALING_EPOCHS,
                'updates_per_model': EXPECTED_UPDATES, 'input_size': [1, 128, 128]}
    for field, value in expected.items():
        if saved_config.get(field) != value:
            raise ValueError(f'保存設定と現在の設定が異なります：{field}。保存時の設定を確認してください。')
    saved_split = read_json(RUN_DIR / 'split_manifest.json')
    if saved_split != split_record:
        raise ValueError('保存済みデータ分割と一致しません。')
    for key in MODEL_KEYS:
        directory = RUN_DIR / key
        model_paths[key] = directory / 'model_final.pth'
        summaries[key] = read_json(directory / 'summary.json')
        if sha256(model_paths[key]) != summaries[key]['weights_sha256']:
            raise ValueError(f'重みが変更されています：{key}')
        if summaries[key]['data_signature'] != data_signature or summaries[key]['updates'] != EXPECTED_UPDATES:
            raise ValueError(f'画像分割・更新回数が異なります：{key}')
        histories[key] = pd.read_csv(directory / 'history.csv')
        if len(histories[key]) != EPOCHS or histories[key].iloc[-1]['cumulative_updates'] != EXPECTED_UPDATES:
            raise ValueError(f'学習ログが不完全です：{key}')
    if len({s['sample_order_sha256'] for s in summaries.values()}) != 1:
        raise ValueError('モデル間で学習画像の順序が異なります。')
    print('比較対象：', RUN_DIR.name, '/ 最終epochのみ / 検証43枚 / testは未使用')
else:
    print('完了済みモデルがありません。学習後に比較できます。')

## 7. 学習曲線を比較する
全モデルで同じepochが同じ更新回数に対応します。左は学習時のMSE、右は未学習の検証画像のMSEです。学習はサンプリング、検証は平均復元です。2つ目の図には各モデルのSSE・KL・総損失を示します。対数軸では0付近を表示下限に置き換えますが、CSVの値は変更しません。

In [ ]:
def plot_learning_curves(history_map, destination=None, show=True):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True, constrained_layout=True)
    for key in MODEL_KEYS:
        h = history_map[key]
        axes[0].plot(h.epoch, h.train_mse, color=COLORS[key], label=LABELS[key])
        axes[1].plot(h.epoch, h.val_mse, color=COLORS[key], label=LABELS[key])
    for ax, title in zip(axes, ['学習画像170枚（サンプリング）', '検証画像43枚（平均復元）']):
        ax.set(title=title, xlabel='エポック', ylabel='MSE / 画素（小さいほどよい）')
        ax.grid(ls='--', alpha=.5)
        ax.legend()
    if destination is not None:
        fig.savefig(destination / 'train_validation_mse.png', dpi=160)
    if show:
        plt.show()
    plt.close(fig)
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    for ax, key in zip(axes.flat, MODEL_KEYS):
        h = history_map[key]
        for column, label in [('train_sse_per_image', '復元SSE'), ('train_kl_per_image', 'KL'),
                              ('train_total_per_image', '総損失（SSE＋βKL）')]:
            ax.plot(h.epoch, np.maximum(h[column], 1e-12), label=label)
        ax.set(title=LABELS[key], xlabel='エポック', ylabel='1画像あたりの値（対数軸）', yscale='log')
        ax.grid(ls='--', alpha=.5)
        ax.legend()
    if destination is not None:
        fig.savefig(destination / 'loss_components.png', dpi=160)
    if show:
        plt.show()
    plt.close(fig)


if READY:
    plot_learning_curves(histories)
else:
    print('学習完了後に曲線を表示できます。')

## 8. 保存した最終重みを読み直し、共通の検証43枚を復元する
4モデルとも平均μから復元します。MSEは画素単位、SSIMは通常の画像単位スコアを計算して43枚で平均します。SSIMは大きいほど入力との構造が似ています。異常検知の1−SSIM上位130点スコアは使いません。

以下で得た復元画像を、そのまま後の図にも使います。図ごとに再推論は行いません。評価結果は実行ごとの新規フォルダへ保存します。

In [ ]:
comparison = pd.DataFrame()
reconstruction_cache, image_metrics = {}, {}
RESULT_DIR = None
before_weight_hashes = {}
if READY:
    rows = []
    for key in MODEL_KEYS:
        before_weight_hashes[key] = sha256(model_paths[key])
        model = MODEL_FACTORIES[key](latent_dim=LATENT_DIM).to(DEVICE)
        model.load_state_dict(torch.load(model_paths[key], map_location=DEVICE, weights_only=True), strict=True)
        values = reconstruct_normal(model, validation_tensor, include_ssim=True)
        reconstruction_cache[key] = values
        image_metrics[key] = validation_frame(values)
        rows.append({'model': key, 'label': LABELS[key], 'epochs': EPOCHS, 'updates': EXPECTED_UPDATES,
                     'parameters': int(model_table.set_index('model').loc[key, 'パラメータ数']),
                     'val_mse': float(values['mse'].mean()), 'val_ssim': float(values['ssim'].mean()),
                     'val_sse_per_image': float(values['sse'].mean()),
                     'val_kl_per_image': float(values['kl'].mean()),
                     'recorded_training_val_mse': summaries[key]['val_mse'],
                     'weight_sha256': before_weight_hashes[key]})
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    comparison = pd.DataFrame(rows)
    evaluation_id = datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
    RESULT_DIR = RUN_DIR / 'evaluations' / evaluation_id
    RESULT_DIR.mkdir(parents=True, exist_ok=False)
    print('比較結果の保存先：', RESULT_DIR)
else:
    print('完了済みモデルがないため、復元品質の比較はまだありません。')

## 9. 復元品質の比較表とグラフ
主な選定基準は検証MSEです。SSIM、実際の端子や背景の復元も併せて判断します。背景の画素が多いため、画像全体のMSEが小さいだけで細い端子が十分に復元されているとは限りません。

In [ ]:
selected_model = None
if READY:
    ranked = comparison.sort_values(['val_mse', 'val_ssim'], ascending=[True, False], kind='stable')
    selected_model = ranked.iloc[0]['model']
    display(ranked[['label', 'parameters', 'updates', 'val_mse', 'val_ssim',
                    'val_sse_per_image', 'val_kl_per_image']].rename(columns={
        'label': 'モデル', 'parameters': 'パラメータ数', 'updates': '更新回数',
        'val_mse': '検証MSE/画素', 'val_ssim': '検証SSIM',
        'val_sse_per_image': '検証SSE/画像', 'val_kl_per_image': '検証KL/画像'}))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    ordered = comparison.set_index('model').loc[list(MODEL_KEYS)]
    positions = np.arange(len(MODEL_KEYS))
    for ax, metric, title in zip(axes, ['val_mse', 'val_ssim'],
                                ['検証MSE：小さいほどよい', '検証SSIM：大きいほどよい']):
        values = ordered[metric].to_numpy()
        ax.bar(positions, values, color=[COLORS[key] for key in MODEL_KEYS])
        ax.set_xticks(positions, [LABELS[key] for key in MODEL_KEYS], rotation=10)
        ax.set_title(title)
        ax.set_ylabel(metric)
        ax.grid(axis='y', ls='--', alpha=.5)
        ax.set_axisbelow(True)
        for x, value in zip(positions, values):
            ax.annotate(f'{value:.6f}' if metric == 'val_mse' else f'{value:.4f}',
                        (x, value), xytext=(0, 5), textcoords='offset points',
                        ha='center', fontsize=10)
        ax.margins(y=.2)
    fig.savefig(RESULT_DIR / 'validation_mse_ssim.png', dpi=160)
    plt.show()
    plt.close(fig)
    display(Markdown(f'この条件で検証MSEが最も小さい候補は **{LABELS[selected_model]}** です。'
                     ' これは今回の正常画像の復元品質による候補であり、異常検知モデルとしての最良を意味しません。'))
else:
    print('比較表・グラフは学習後に表示します。')

## 10. 同じ画像の入力・復元・絶対誤差と、端子の拡大
初期設定では検証画像の先頭6枚を表示します。MAX_RECON_IMAGES=Noneですべての43枚を表示できます。画像順はモデル間で共通です。誤差マップは入力と復元の絶対差（0〜1）で、欠陥の正解位置やSSIM異常度ではありません。カラースケールは全図で共通です。

端子の拡大範囲はLEAD_ROIで変更できます。これは固定した表示範囲であり、部品の位置を自動検出するものではありません。図は評価時に保存したメモリ上の復元結果を使用し、再推論しません。

In [ ]:
if READY:
    count = len(validation_manifest) if MAX_RECON_IMAGES is None else min(MAX_RECON_IMAGES, len(validation_manifest))
    qualitative_dir = RESULT_DIR / 'reconstructions'
    qualitative_dir.mkdir(exist_ok=True)
    print(f'検証{len(validation_manifest)}枚中{count}枚を表示します。')
    for index in range(count):
        record = validation_manifest[index]
        original = validation_tensor[index, 0].numpy()
        fig, axes = plt.subplots(len(MODEL_KEYS), 3, figsize=(10, 12), squeeze=False)
        fig.subplots_adjust(left=.1, right=.85, top=.91, bottom=.03, hspace=.45, wspace=.3)
        for row, key in enumerate(MODEL_KEYS):
            reconstructed = reconstruction_cache[key]['reconstruction'][index]
            absolute_error = np.abs(original - reconstructed)
            for col, array in enumerate([original, reconstructed]):
                axes[row, col].imshow(array, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
            heat = axes[row, 2].imshow(absolute_error, cmap='inferno', vmin=0,
                                      vmax=ABS_ERROR_VMAX, interpolation='nearest')
            axes[row, 0].set_title('入力（復元目標）')
            axes[row, 0].set_ylabel(LABELS[key], fontsize=10)
            mse_value = reconstruction_cache[key]['mse'][index]
            ssim_value = reconstruction_cache[key]['ssim'][index]
            axes[row, 1].set_title(f'復元画像\nMSE={mse_value:.6f} / SSIM={ssim_value:.4f}', fontsize=10)
            axes[row, 2].set_title('絶対誤差 |入力−復元|', fontsize=10)
            for ax in axes[row]:
                ax.set_xticks([])
                ax.set_yticks([])
        colorbar_axis = fig.add_axes([.88, .06, .025, .8])
        fig.colorbar(heat, cax=colorbar_axis, label='画素値の絶対差（全モデル共通）')
        fig.suptitle(f'{record["path"]} / 検証用正常画像 / 平均μから復元', fontsize=12)
        stem = f'{index:02d}_{Path(record["path"]).stem}'
        fig.savefig(qualitative_dir / f'{stem}_input_reconstruction_error.png', dpi=160)
        plt.show()
        plt.close(fig)
        fig, axes = plt.subplots(1, len(MODEL_KEYS) + 1, figsize=(15, 4), constrained_layout=True)
        arrays = [original] + [reconstruction_cache[key]['reconstruction'][index] for key in MODEL_KEYS]
        titles = ['入力の端子領域'] + [LABELS[key] for key in MODEL_KEYS]
        for ax, array, title in zip(axes, arrays, titles):
            ax.imshow(array[ry0:ry1, rx0:rx1], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
            ax.set_title(title, fontsize=10)
            ax.axis('off')
        fig.suptitle(f'{record["path"]} / 同じ端子領域 {LEAD_ROI}', fontsize=12)
        fig.savefig(qualitative_dir / f'{stem}_lead_zoom.png', dpi=160)
        plt.show()
        plt.close(fig)
else:
    print('モデル評価後に入出力と端子の拡大を表示します。')

## 11. 比較結果と識別情報を保存する
選定結果は検証MSEを主基準にした候補です。同率の場合のみSSIMで並べます。元データ・保存済み重みは変更せず、評価結果の新規フォルダへ保存します。

In [ ]:
if READY:
    for key in MODEL_KEYS:
        if sha256(model_paths[key]) != before_weight_hashes[key]:
            raise AssertionError(f'評価中に重みが変更されています：{key}')
    for record in manifest:
        if sha256(ROOT / record['path']) != record['file_sha256']:
            raise AssertionError(f'元画像が変更されています：{record["path"]}')
    comparison.to_csv(RESULT_DIR / 'comparison.csv', index=False)
    per_image = pd.concat([frame.assign(model=key) for key, frame in image_metrics.items()], ignore_index=True)
    per_image.to_csv(RESULT_DIR / 'validation_per_image.csv', index=False)
    plot_learning_curves(histories, destination=RESULT_DIR, show=False)
    evaluation_record = {'experiment': EXPERIMENT, 'training_run_id': RUN_DIR.name,
                         'evaluation_id': RESULT_DIR.name,
                         'created_jst': datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),
                         'data_signature': data_signature, 'validation_manifest': validation_manifest,
                         'inference': 'posterior mean; model.eval(); no sampling',
                         'mse': 'pixel SSE / (128*128), averaged across validation images',
                         'ssim': {'window': SSIM_WINDOW, 'data_range': 1.0, 'top_k_used': False},
                         'selected_model_by_validation_mse': selected_model,
                         'selection_note': 'final epoch only; lowest MSE, SSIM only as exact-MSE tie breaker',
                         'test_data_used': False, 'seed': TRAIN_SEED, 'device': str(DEVICE),
                         'python': platform.python_version(), 'torch_version': torch.__version__,
                         'training_config': saved_config, 'comparison': comparison.to_dict('records'),
                         'visualization': {'max_images': MAX_RECON_IMAGES, 'lead_roi': list(LEAD_ROI),
                                           'absolute_error_vmax': ABS_ERROR_VMAX}}
    write_json(RESULT_DIR / 'evaluation_record.json', evaluation_record)
    print('比較結果を保存しました：', RESULT_DIR)
else:
    print('完了済みモデルがないため、正式な比較結果は保存しません。')

## 結果を見た後に検討すること
- これは「正常を復元するモデル」の候補選びです。異常まで復元しやすくなる可能性もあるため、異常検知の最終評価は別途AUROCで行います。
- 検証43枚で候補を選んだ後、必要なら正常213枚すべてで選んだモデルを再学習する設計を別途決めます。今回はtestを使用しません。
- 全モデルの復元が不十分なら、構造だけが原因とは限りません。βと潜在情報の維持、潜在次元、学習量、128入力で失う細部なども次の検討事項です。このノートブックはそれらを同時に変更しません。
- MSEが低くても端子がぼやけている場合があります。検証SSIM・端子の拡大・複数画像の見た目を一緒に確認します。
- 同じseedでも、構造が異なれば初期重みや学習の進み方は異なります。今回の4モデル・単一seed・固定条件の範囲での比較です。
- 過去のtransistor実験は学習213枚をすべて使っているため、今回の170/43分割の数値とそのまま比較しません。

参照モデル：[7月の4モデル比較](../202607/260709AUROCを用いた比較評価.ipynb)、[transistorの通常／reflect回転比較](261105transistorの通常学習とreflect回転拡張の比較学習とAUROC評価.ipynb)。既存ノートブック・学習結果は変更しません。